# Notebook for merging the datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
#mouting the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive/TFG/oulad

/content/drive/MyDrive/TFG/oulad


In [ ]:
vle = pd.read_csv('vle.csv')
studentVle = pd.read_csv('studentVle.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')
studentInfo = pd.read_csv('studentInfo.csv')
studentAssessment = pd.read_csv('studentAssessment.csv')
courses = pd.read_csv('courses.csv')
assessments = pd.read_csv('assessments.csv')

##Merging 

Hay 4 casillas donde la continua sea nulo y el examen tenga nota.
conclusiones: esos un 0
4955 casillas con las dos completas:
conclusiones: dejamos como está
20884 donde el examen sea nulo y la continua no
conclusiones: CCC Y DDD un 0 en el examen, el resto replicamos la nota de la continua ya que no existen los examenes
6750 con los dos valores nulos:
conclusiones: los dos con un 0 

mean clicks y total clicks nans=los ponemos en 0 y punto
los mean score por tma, cma, exam cuando no hay: sustituir por otro valor (su media de otro)
date registration: nans a 0
date unregistration: nans al ultimo día del curso (duración curso)
imbd band=comparar.

Añadir por tipo de recurso los recursos, 0 para cuando no hay (añadir solo la suma total)
mean clicks: mean clicks por día mejor?? ver que hacer, si poner uno que sea total clicks/total dias o similar?

recordatorio: cambiar los weigh

POR PASOS
Paso 1: cambiar los weights de GGG en assesments HECHO

Paso 2: Meter una columna que sea sum_clicks/duración curso HECHO

Paso 3: Sustituir NANS de unregistration date por el día final del curso HECHO

Paso 4: Añadir la columna de sum_clicks totales por tipo de recurso HECHO

Paso 5: Poner en 0 los nans de registration HECHO

Paso 6: Poner los nulos en la continua con un 0

Paso 7: Nota examen CCC Y DDD un 0 en el examen, el resto replicamos la nota de la continua ya que no existen los examenes

Paso 8: los mean score por tma, cma, exam cuando no hay: sustituir por otro valor (su media de otro)

Paso 9: imbd band=comparar.


In [ ]:
df_student=pd.merge(studentRegistration, studentInfo, on=['id_student', 'code_module', 'code_presentation'])

In [ ]:
df_vle=pd.merge(vle, studentVle, on=['id_site', 'code_module', 'code_presentation'])

In [ ]:
df_vle.head()

,id_site,code_module,code_presentation,activity_type,week_from,week_to,id_student,date,sum_click
0,546943,AAA,2013J,resource,NaN,NaN,75091,-10,1
1,546943,AAA,2013J,resource,NaN,NaN,186149,-10,1
2,546943,AAA,2013J,resource,NaN,NaN,205350,-10,2
3,546943,AAA,2013J,resource,NaN,NaN,1626710,-9,1
4,546943,AAA,2013J,resource,NaN,NaN,2643002,-8,1


In [ ]:
df_vle_mean=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.mean().reset_index(name='mean_clicks')

In [ ]:
df_vle_sum=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.sum().reset_index(name='total_clicks')

In [ ]:
df_vle_mean_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.mean().reset_index(name='mean_clicks')

In [ ]:
df_vle_sum_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.sum().reset_index(name='sum_clicks')

In [ ]:
df_vle_sum_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'activity_type']).sum_click.sum().reset_index(name='total_clicks')

In [ ]:
df_vle_sum_type.head()

,id_student,code_module,code_presentation,activity_type,total_clicks
0,6516,AAA,2014J,dataplus,21
1,6516,AAA,2014J,forumng,451
2,6516,AAA,2014J,homepage,497
3,6516,AAA,2014J,oucontent,1505
4,6516,AAA,2014J,resource,31


In [ ]:
activities=list(df_vle_sum_type.activity_type.unique())
for activity in activities:
    df_sum_x=df_vle_sum_type[df_vle_sum_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'total_clicks': 'clicks_from_' + str(activity)})
    df_vle_sum=pd.merge(df_vle_sum, df_sum_x, on =['id_student', 'code_module', 'code_presentation'], how = 'outer')

In [ ]:
df_vle_sum_by_date_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date', 'activity_type']).sum_click.sum().reset_index(name='sum_clicks')

In [ ]:
activities=list(df_vle_sum_by_date_type.activity_type.unique())
for activity in activities:
    df_sum_x=df_vle_sum_by_date_type[df_vle_sum_by_date_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'sum_clicks': 'clicks_from_' + str(activity)})
    df_vle_sum_by_date=pd.merge(df_vle_sum_by_date, df_sum_x, on =['id_student', 'code_module', 'code_presentation', 'date'], how = 'outer')

In [ ]:
df_vle_sum_by_date.head()

,id_student,code_module,code_presentation,date,sum_clicks,clicks_from_homepage,clicks_from_oucontent,clicks_from_subpage,clicks_from_forumng,clicks_from_resource,...,clicks_from_quiz,clicks_from_page,clicks_from_glossary,clicks_from_ouelluminate,clicks_from_questionnaire,clicks_from_dualpane,clicks_from_folder,clicks_from_htmlactivity,clicks_from_sharedsubpage,clicks_from_repeatactivity
0,6516,AAA,2014J,-23,28,3.0,23.0,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6516,AAA,2014J,-22,82,13.0,34.0,NaN,33.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6516,AAA,2014J,-20,41,12.0,8.0,1.0,13.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6516,AAA,2014J,-17,7,2.0,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6516,AAA,2014J,-12,2,1.0,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
assessments.loc[(assessments['code_module'] == 'GGG') & (assessments['assessment_type'] != 'Exam'), 'weight']=11

In [ ]:
df_assessments=pd.merge(studentAssessment, assessments, on=['id_assessment'])

In [ ]:
df_assesments_score_mean=df_assessments.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='mean_score')

In [ ]:
df_assessments_tma=df_assessments[(df_assessments['assessment_type']=='TMA')]
df_assesments_tma_score_mean=df_assessments_tma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='tma_mean_score')

In [ ]:
df_assessments_cma=df_assessments[(df_assessments['assessment_type']=='CMA')]
df_assesments_cma_score_mean=df_assessments_cma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='cma_mean_score')

In [ ]:
df_assessments_exam=df_assessments[(df_assessments['assessment_type']=='Exam')]
df_assesments_exam_score_mean=df_assessments_exam.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='exam_mean_score')

In [ ]:
df_score_mean_1=pd.merge(df_assesments_tma_score_mean, df_assesments_cma_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_score_mean=pd.merge(df_score_mean_1, df_assesments_exam_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_assessments_mean=pd.merge(df_score_mean, df_assesments_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_final_grade=df_assessments[['id_student', 'score', 'code_module','code_presentation', 'assessment_type', 'weight']]

In [ ]:
df_final_grade['weighted']=df_final_grade.score * df_final_grade.weight * 0.01

<ipython-input-28-c56330c897a2>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_grade['weighted']=df_final_grade.score * df_final_grade.weight * 0.01


In [ ]:
df_final_grade_cont=df_final_grade[(df_final_grade['assessment_type']!='Exam')]
df_final_grade_cont=df_final_grade_cont.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='continuous_grade')

In [ ]:
df_final_grade_exam=df_final_grade[(df_final_grade['assessment_type']=='Exam')]
df_final_grade_exam=df_final_grade_exam.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='exam_grade')

In [ ]:
df_final=df_student

In [ ]:
df_final=pd.merge(df_final, df_assessments_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_final=pd.merge(df_final, df_vle_sum, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_final=pd.merge(df_final, df_vle_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_final=pd.merge(df_final, df_final_grade_cont, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
df_final=pd.merge(df_final, df_final_grade_exam, on=['id_student', 'code_module', 'code_presentation'], how="outer")

In [ ]:
#df_final.to_csv('df_final_st.csv')

In [ ]:
pd.options.display.max_columns = None

In [ ]:
courses.head()

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


In [1]:
df_final=pd.merge(df_final, courses, on=['code_module', 'code_presentation'], how="outer")

NameError: ignored

In [ ]:
df_final['mean_clicks_per_day']=df_final['mean_clicks']/df_final['module_presentation_length']

In [ ]:
df_final.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,tma_mean_score,cma_mean_score,exam_mean_score,mean_score,total_clicks,clicks_from_dataplus,clicks_from_forumng,clicks_from_homepage,clicks_from_oucontent,clicks_from_resource,clicks_from_subpage,clicks_from_url,clicks_from_externalquiz,clicks_from_oucollaborate,clicks_from_ouwiki,clicks_from_quiz,clicks_from_page,clicks_from_glossary,clicks_from_ouelluminate,clicks_from_dualpane,clicks_from_folder,clicks_from_questionnaire,clicks_from_htmlactivity,clicks_from_sharedsubpage,clicks_from_repeatactivity,mean_clicks,continuous_grade,exam_grade,module_presentation_length,mean_clicks_per_day
0,AAA,2013J,11391,-159.0,NaN,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,NaN,NaN,82.0,934.0,NaN,193.0,138.0,553.0,13.0,32.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.765306,82.4,NaN,268,0.017781
1,AAA,2013J,28400,-53.0,NaN,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,NaN,NaN,66.4,1435.0,10.0,417.0,324.0,537.0,12.0,87.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.337209,65.4,NaN,268,0.012452
2,AAA,2013J,30268,-92.0,12.0,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,NaN,NaN,281.0,NaN,126.0,59.0,66.0,4.0,22.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.697368,NaN,NaN,268,0.013796
3,AAA,2013J,31604,-52.0,NaN,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,NaN,NaN,76.0,2158.0,2.0,634.0,432.0,836.0,19.0,144.0,90.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.254902,76.3,NaN,268,0.012145
4,AAA,2013J,32885,-176.0,NaN,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,NaN,NaN,54.4,1034.0,NaN,194.0,204.0,494.0,45.0,79.0,14.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.937500,55.0,NaN,268,0.010961


In [ ]:
df_final['date_registration'] = df_final['date_registration'].fillna(0)

In [ ]:
df_final.loc[pd.isna(df_final["date_unregistration"]), "date_unregistration"] = df_final[pd.isna(df_final["date_unregistration"])]["module_presentation_length"].apply(
    lambda x: x
)

In [ ]:
df_final['date_registration'] = df_final['date_registration'].fillna(0)

In [ ]:
df_final.iloc[:,17:41]=df_final.iloc[:,17:41].fillna(0)

In [ ]:
def condition(x):
    if x=='CCC':
      return 0
    if x=='DDD':
      return 0
    else:
      return

In [ ]:
df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["code_module"].apply(condition)

In [ ]:
df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["continuous_grade"].apply(lambda x: x)

Paso 7: Nota examen CCC Y DDD un 0 en el examen, el resto replicamos la nota de la continua ya que no existen los examenes

In [ ]:
df_final['mean_clicks_per_day'] = df_final['mean_clicks_per_day'].fillna(0)

In [ ]:
df_final.isnull().sum()

code_module                       0
code_presentation                 0
id_student                        0
date_registration                 0
date_unregistration               0
gender                            0
region                            0
highest_education                 0
imd_band                       1111
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
tma_mean_score                 7805
cma_mean_score                17493
exam_mean_score               27634
mean_score                        0
total_clicks                      0
clicks_from_dataplus              0
clicks_from_forumng               0
clicks_from_homepage              0
clicks_from_oucontent             0
clicks_from_resource              0
clicks_from_subpage               0
clicks_from_url                   0
clicks_from_externalquiz          0
clicks_from_oucollaborate   

In [ ]:
df_final['imd_band'].value_counts()

20-30%     3654
30-40%     3539
10-20      3516
0-10%      3311
40-50%     3256
50-60%     3124
60-70%     2905
70-80%     2879
80-90%     2762
90-100%    2536
Name: imd_band, dtype: int64

In [ ]:
df_final.final_result.value_counts()

Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: final_result, dtype: int64

In [ ]:
df_final.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,tma_mean_score,cma_mean_score,exam_mean_score,mean_score,total_clicks,clicks_from_dataplus,clicks_from_forumng,clicks_from_homepage,clicks_from_oucontent,clicks_from_resource,clicks_from_subpage,clicks_from_url,clicks_from_externalquiz,clicks_from_oucollaborate,clicks_from_ouwiki,clicks_from_quiz,clicks_from_page,clicks_from_glossary,clicks_from_ouelluminate,clicks_from_dualpane,clicks_from_folder,clicks_from_questionnaire,clicks_from_htmlactivity,clicks_from_sharedsubpage,clicks_from_repeatactivity,mean_clicks,continuous_grade,exam_grade,module_presentation_length,mean_clicks_per_day
0,AAA,2013J,11391,-159.0,268.0,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,NaN,NaN,82.0,934.0,0.0,193.0,138.0,553.0,13.0,32.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.765306,82.4,82.4,268,0.017781
1,AAA,2013J,28400,-53.0,268.0,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,NaN,NaN,66.4,1435.0,10.0,417.0,324.0,537.0,12.0,87.0,48.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.337209,65.4,65.4,268,0.012452
2,AAA,2013J,30268,-92.0,12.0,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,NaN,0.0,281.0,0.0,126.0,59.0,66.0,4.0,22.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.697368,0.0,0.0,268,0.013796
3,AAA,2013J,31604,-52.0,268.0,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,NaN,NaN,76.0,2158.0,2.0,634.0,432.0,836.0,19.0,144.0,90.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.254902,76.3,76.3,268,0.012145
4,AAA,2013J,32885,-176.0,268.0,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,NaN,NaN,54.4,1034.0,0.0,194.0,204.0,494.0,45.0,79.0,14.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.937500,55.0,55.0,268,0.010961


In [ ]:
df_final.to_csv('df_v2.csv')